<a href="https://colab.research.google.com/github/frayyan-rgb/amazon-employee-feedback-analysis/blob/main/10_friction_table.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

themes = pd.read_csv("combined_with_themes.csv")
seg = pd.read_csv("Project5_Labeled_Segments.csv")[["review_id", "Segment"]]

df = themes[themes["platform"] == "Glassdoor"].copy()
df["review_id"] = df["review_id"].astype(int)
df = df.merge(seg, on="review_id", how="left")

In [2]:
df["Theme"] = df["Theme"].fillna("")
ft = df[df["Segment"] == "Full-Time Associates"].copy()
print(len(ft))   # expect 77

77


In [3]:
RATE_WORDS = r"\brates?\b|write.?ups?|numbers|targets?|quota|productivity|metrics?"
ft["Rate Pressure"] = ft["review_cons"].fillna("").str.contains(RATE_WORDS, case=False, regex=True)

In [4]:
n_shift = ft["Theme"].str.contains("Shift Strain", regex=False).sum()
n_burn  = ft["Theme"].str.contains("Burnout & Fatigue", regex=False).sum()
n_rate  = ft["Rate Pressure"].sum()
print(n_shift, n_burn, n_rate)   # expect 16 12 12

16 12 12


In [5]:
def show(mask, n=8):
    for _, r in ft[mask].head(n).iterrows():
        print(r["review_id"], "|", str(r["review_cons"])[:140].replace("\n", " / "))

show(ft["Theme"].str.contains("Shift Strain", regex=False))
show(ft["Rate Pressure"])

7 | fast paced working environment, weight lifting
9 | management absolutely is terrible at communicating
11 | standing +10 hours / noise / safety hazards  / overworking / get write ups for not meeting their rate expectations
13 | night-shift / overtime / no cons at all
20 | there is favoritism- like any other job.
26 | long hours, minimal light, will write you up for bathroom breaks, mandatory overtime with little to no heads up. this place drains you emoti
30 | monotonous work / not particularly rewarding work / work can become slow from january to may
66 | metrics are always a part of their work experience. day to day optimizations, pas always coming by to give you feedback can make it seem lik
11 | standing +10 hours / noise / safety hazards  / overworking / get write ups for not meeting their rate expectations
32 | too much physical work with not much of job security / pressure for target work negatively impact physical and mental health / biased leadership
48 | work-life balance,

In [6]:
rows = [
 ("Full-Time Associates",
  "Works 10-12 hour shifts, with overtime and night hours that can be mandatory",
  "Long shifts and extra hours arrive with little notice, and breaks are limited",
  "Fatigue and a hard time planning life outside work, which likely lowers morale and raises turnover",
  '"mandatory overtime with little to no heads up"',
  f"{n_shift} reviews tagged Shift Strain"),
 ("Full-Time Associates",
  "Does physically demanding, repetitive work, often standing for the whole shift",
  "Little recovery time inside the shift, so strain builds up",
  "Burnout, injury risk, and absence, which likely lead to attrition and slower output late in the shift",
  '"its a physical job so prepare to be tired."',
  f"{n_burn} reviews tagged Burnout & Fatigue"),
 ("Full-Time Associates",
  "Is tracked on rate and targets throughout the shift",
  "Missing rate brings write-ups, and associates feel valued only for their numbers",
  "Pressure to cut corners and quit, which likely raises safety risk, turnover, and rehiring cost",
  '"get write ups for not meeting their rate expectations"',
  f"{n_rate} reviews mention rate, targets, or numbers (keyword search)"),
]
cols = ["Segment", "Experience", "Friction Point", "Impact", "Evidence (quote)", "Evidence (count)"]
friction = pd.DataFrame(rows, columns=cols)

# safety check: every quote must exist in the real reviews
alltext = " ".join(df["review_cons"].fillna("").str.lower())
for q in friction["Evidence (quote)"]:
    assert q.strip('"').lower() in alltext, q

friction

,Segment,Experience,Friction Point,Impact,Evidence (quote),Evidence (count)
0,Full-Time Associates,"Works 10-12 hour shifts, with overtime and nig...",Long shifts and extra hours arrive with little...,Fatigue and a hard time planning life outside ...,"""mandatory overtime with little to no heads up""",16 reviews tagged Shift Strain
1,Full-Time Associates,"Does physically demanding, repetitive work, of...","Little recovery time inside the shift, so stra...","Burnout, injury risk, and absence, which likel...","""its a physical job so prepare to be tired.""",12 reviews tagged Burnout & Fatigue
2,Full-Time Associates,Is tracked on rate and targets throughout the ...,"Missing rate brings write-ups, and associates ...","Pressure to cut corners and quit, which likely...","""get write ups for not meeting their rate expe...","12 reviews mention rate, targets, or numbers (..."


In [7]:
from openpyxl.styles import Alignment

with pd.ExcelWriter("Project5_Friction_Table.xlsx", engine="openpyxl") as w:
    friction.to_excel(w, index=False, sheet_name="Friction Table")
    ws = w.sheets["Friction Table"]
    for col, width in zip("ABCDEF", [24, 40, 40, 48, 40, 34]):
        ws.column_dimensions[col].width = width
    for row in ws.iter_rows():
        for c in row:
            c.alignment = Alignment(wrap_text=True, vertical="top")

In [8]:
from google.colab import files
files.download("Project5_Friction_Table.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>